# Hybrid network-stress inference

This notebook demonstrates the complete inference path: load the saved model,
select a historical prediction origin, make a one-hour-ahead ML prediction,
assert operational facts into Prolog, and return the symbolic interpretation and
recommendations.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

for base in (Path.cwd(), *Path.cwd().parents):
    if (base / "A1" / "src" / "DataLoader.py").is_file():
        PROJECT_ROOT = base
        break
else:
    raise FileNotFoundError("Could not locate the project root.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from A2.src.inference import print_result, run_inference

DATA_PATH = (
    PROJECT_ROOT / "A1" / "data" / "processed"
    / "electricity_transformed.parquet"
)
MODEL_PATH = PROJECT_ROOT / "A2" / "models" / "network_stress_model.joblib"
RULES_PATH = PROJECT_ROOT / "A2" / "src" / "stress_rules.pl"


## Select a prediction origin

Change `TIMESTAMP` to test another historical observation. Set it to `None` to
receive an interactive prompt. `RESERVE_MARGIN` is optional because the parquet
does not contain available capacity; only provide it from an authoritative source.


In [ ]:
TIMESTAMP = "2026-09-23 23:00:00+10:00"
RESERVE_MARGIN = None

if TIMESTAMP is None:
    TIMESTAMP = input("Enter an observation timestamp: ").strip()


In [ ]:
result = run_inference(
    data_path=DATA_PATH,
    model_path=MODEL_PATH,
    rules_path=RULES_PATH,
    timestamp_text=TIMESTAMP,
    reserve_margin=RESERVE_MARGIN,
)

print_result(result)


In [ ]:
display(
    pd.Series(result["probabilities"], name="probability")
    .sort_values(ascending=False)
    .to_frame()
)

pd.DataFrame(
    {
        "stage": ["ML prediction", "Symbolic final risk"],
        "risk": [result["ml_prediction"], result["final_risk"]],
    }
)
